In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

import sys
sys.path.append('../')
from utils import * #only needed for xgboost
from friedman1 import *

import warnings
warnings.filterwarnings("ignore") #just to supress warnings

In [ ]:
#Function that returns best params for each pair of (d, target_instance) 
# and their average val rmse, rmse, mae, given a set of config columns (depending on method)
# This funcion can take a dataframe or a path to a csv as input.
# the best settings are chosen using the lowest val rmse
def topk_per_d_per_method(data, config_cols, k=5):
    """
    Return up to the top-k configs per (target_instances, d), ranked by avg_rmse.
    Averages are computed across seeds.
    """
    df = pd.read_csv(data, index_col=[0]) if isinstance(data, str) else data.copy()
    df = df.sort_values(by=['seed'] + config_cols)

    # aggregate over seeds
    agg = (df.groupby(config_cols, as_index=False)
             .agg(avg_rmse=('rmse', 'mean'),
                  avg_mae=('val_mae', 'mean'),
                  n_seeds=('seed', 'nunique')))

    # merge back on config_cols only (minimal fix)
    agg = agg.merge(df, on=config_cols, how='left')

    # sort globally by the fields that determine the ranking
    agg = agg.sort_values(by=['target_instances', 'd', 'avg_rmse'], ascending=True)

    # select top-k per (target_instances, d)
    topk_df = agg.groupby(['target_instances', 'd'], group_keys=False).head(k)

    # final compact output
    topk_d = topk_df[['target_instances', 'd', 'method', 'avg_rmse', 'rmse', 'mae']]

    return topk_d, topk_df




In [ ]:
#Read data (remove v = 0.05 because we chose not to include it)
data_gaussian = pd.read_csv('results/gaussian.csv')
data_gaussian = data_gaussian[data_gaussian['v'] == 0.1]
data_slash = pd.read_csv('results/slash.csv')
data_slash = data_slash[data_slash['v'] == 0.1]
data_t= pd.read_csv('results/t.csv')
data_t = data_t[data_t['v'] == 0.1]

# Here we make final vizes!

In [ ]:

plt.figure(figsize = (18,18))
config_cols = ['target_instances', 'd', 'v', 'source_tree_size', 'target_tree_size', 'm_0', 'k']
import matplotlib as mpl
# Here we find the optimal settings (top5x5 = 25 total RMSEs)
LS_data_gaussian = data_gaussian[data_gaussian['method'] == 'LSTransferTreeBoost']
LAD_data_gaussian = data_gaussian[data_gaussian['method'] == 'LADTransferTreeBoost']
Huber_data_gaussian = data_gaussian[data_gaussian['method'] == 'MTransferTreeBoost']
LS_data_gaussian['method'] = 'LS'
LAD_data_gaussian['method'] = 'LAD'
Huber_data_gaussian['method'] = 'M'

best_LS, best_LS_params = topk_per_d_per_method(
    LS_data_gaussian,
    config_cols, k=25
)

best_LAD, best_LAD_params = topk_per_d_per_method(
    LAD_data_gaussian,
    config_cols, k=25
)

best_Huber, best_Huber_params = topk_per_d_per_method(
    Huber_data_gaussian,
    config_cols, k=25
)

i = 1

# create new df for viz
df = pd.concat([best_LS, best_LAD])
df = pd.concat([df, best_Huber])

for target_instances in [100, 300, 500]:
    plt.subplot(3,3,i)
    ax = sns.boxplot(x='d', y='rmse', hue='method', data=df[df['target_instances'] == target_instances], showfliers=False)
    plt.title('Gaussian', fontsize = 14)
    plt.ylabel('RMSE', fontsize=12)
    plt.xlabel('d', fontsize=12)
    # hatches must equal the number of hues (3 in this case)
    hatches = ['//', '//', '//', '..', '..', '..', 'xx','xx','xx']
    # select the correct patches
    patches = [patch for patch in ax.patches if type(patch) == mpl.patches.PathPatch]
    # the number of patches should be evenly divisible by the number of hatches
    h = hatches * (len(patches) // len(hatches))
    # iterate through the patches for each subplot
    for patch, hatch in zip(patches, h):
        patch.set_hatch(hatch)
        fc = patch.get_facecolor()
        patch.set_edgecolor(fc)
        patch.set_facecolor('none')

    l = ax.legend()
    hatches_in_plot = ['//', '..', 'xx']
    for lp, hatch in zip(l.get_patches(), hatches_in_plot):
        lp.set_hatch(hatch)
        fc = lp.get_facecolor()
        lp.set_edgecolor(fc)
        lp.set_facecolor('none')
    i += 3



i = 2

# Here we find the optimal settings
LS_data_slash = data_slash[data_slash['method'] == 'LSTransferTreeBoost']
LAD_data_slash = data_slash[data_slash['method'] == 'LADTransferTreeBoost']
Huber_data_slash = data_slash[data_slash['method'] == 'MTransferTreeBoost']
LS_data_slash['method'] = 'LS'
LAD_data_slash['method'] = 'LAD'
Huber_data_slash['method'] = 'M'

best_LS, best_LS_params = topk_per_d_per_method(
    LS_data_slash,
    config_cols, k=25
)

best_LAD, best_LAD_params = topk_per_d_per_method(
    LAD_data_slash,
    config_cols, k=25
)

best_Huber, best_Huber_params = topk_per_d_per_method(
    Huber_data_slash,
    config_cols, k=25
)

# create new df for viz
df = pd.concat([best_LS, best_LAD])
df = pd.concat([df, best_Huber])
for target_instances in [100, 300, 500]:
    plt.subplot(3,3,i)
    ax = sns.boxplot(x='d', y='rmse', hue='method', data=df[df['target_instances'] == target_instances], showfliers=False)
    plt.title('Slash', fontsize = 14)
    plt.ylabel('RMSE', fontsize=12)
    plt.xlabel('d', fontsize=12)
    # hatches must equal the number of hues (3 in this case)
    hatches = ['//', '//', '//', '..', '..', '..', 'xx','xx','xx']
    # select the correct patches
    patches = [patch for patch in ax.patches if type(patch) == mpl.patches.PathPatch]
    # the number of patches should be evenly divisible by the number of hatches
    h = hatches * (len(patches) // len(hatches))
    # iterate through the patches for each subplot
    for patch, hatch in zip(patches, h):
        patch.set_hatch(hatch)
        fc = patch.get_facecolor()
        patch.set_edgecolor(fc)
        patch.set_facecolor('none')

    l = ax.legend()
    hatches_in_plot = ['//', '..', 'xx']
    for lp, hatch in zip(l.get_patches(), hatches_in_plot):
        lp.set_hatch(hatch)
        fc = lp.get_facecolor()
        lp.set_edgecolor(fc)
        lp.set_facecolor('none')
    i += 3


i = 3
# Here we find the optimal settings
LS_data_t = data_t[data_t['method'] == 'LSTransferTreeBoost']
LAD_data_t = data_t[data_t['method'] == 'LADTransferTreeBoost']
Huber_data_t = data_t[data_t['method'] == 'MTransferTreeBoost']
LS_data_t['method'] = 'LS'
LAD_data_t['method'] = 'LAD'
Huber_data_t['method'] = 'M'

best_LS, best_LS_params = topk_per_d_per_method(
    LS_data_t,
    config_cols, k=25
)

best_LAD, best_LAD_params = topk_per_d_per_method(
    LAD_data_t,
    config_cols, k=25
)

best_Huber, best_Huber_params = topk_per_d_per_method(
    Huber_data_t,
    config_cols, k=25
)

# create new df for viz
df = pd.concat([best_LS, best_LAD])
df = pd.concat([df, best_Huber])
for target_instances in [100, 300, 500]:
    plt.subplot(3,3,i)
    ax = sns.boxplot(x='d', y='rmse', hue='method', data=df[df['target_instances'] == target_instances], showfliers=False)
    plt.title('$t(2)$', fontsize = 14)
    plt.ylabel('RMSE', fontsize=12)
    plt.xlabel('d', fontsize=12)
    # hatches must equal the number of hues (3 in this case)
    hatches = ['//', '//', '//', '..', '..', '..', 'xx','xx','xx']
    # select the correct patches
    patches = [patch for patch in ax.patches if type(patch) == mpl.patches.PathPatch]
    # the number of patches should be evenly divisible by the number of hatches
    h = hatches * (len(patches) // len(hatches))
    # iterate through the patches for each subplot
    for patch, hatch in zip(patches, h):
        patch.set_hatch(hatch)
        fc = patch.get_facecolor()
        patch.set_edgecolor(fc)
        patch.set_facecolor('none')

    l = ax.legend()
    hatches_in_plot = ['//', '..', 'xx']
    for lp, hatch in zip(l.get_patches(), hatches_in_plot):
        lp.set_hatch(hatch)
        fc = lp.get_facecolor()
        lp.set_edgecolor(fc)
        lp.set_facecolor('none')
    i += 3

plt.subplots_adjust(hspace=0.2)  # increase horizontal spacing
plt.savefig('vizes/ls_lad_huber.png', bbox_inches = 'tight', pad_inches = 0.05, dpi = 300)


In [ ]:
# Here we find the optimal settings for gaussian (and each no. of target instances and d-value)
config_cols = ['v', 'source_tree_size', 'target_tree_size', 'm_0', 'k']
target_instances_list = [100, 300, 500]
d_list = [3,6,9]
for target_instances in target_instances_list:
    for d in d_list:
        LS_data_gaussian = data_gaussian[data_gaussian['method'] == 'LSTransferTreeBoost']
        LS_data_gaussian = LS_data_gaussian[(LS_data_gaussian['d'] == d) & (LS_data_gaussian['target_instances'] == target_instances)]

        best_LS, best_LS_params = topk_per_d_per_method(
            LS_data_gaussian,
            config_cols, k=1000
        )
        best_LS_params = best_LS_params.iloc[range(0, len(best_LS_params), 5)]
        best_LS_params.to_csv(f'results/optimal_params_LS_gaussian_{target_instances}_{d}.csv')

        LAD_data_gaussian = data_gaussian[data_gaussian['method'] == 'LADTransferTreeBoost']
        LAD_data_gaussian = LAD_data_gaussian[(LAD_data_gaussian['d'] == d) & (LAD_data_gaussian['target_instances'] == target_instances)]

        best_LAD, best_LAD_params = topk_per_d_per_method(
            LAD_data_gaussian,
            config_cols, k=1000
        )
        best_LAD_params = best_LAD_params.iloc[range(0, len(best_LAD_params), 5)]
        best_LAD_params.to_csv(f'results/optimal_params_LAD_gaussian_{target_instances}_{d}.csv')

        M_data_gaussian = data_gaussian[data_gaussian['method'] == 'MTransferTreeBoost']
        M_data_gaussian = M_data_gaussian[(M_data_gaussian['d'] == d) & (M_data_gaussian['target_instances'] == target_instances)]

        best_M, best_M_params = topk_per_d_per_method(
            M_data_gaussian,
            config_cols, k=1000
        )
        best_M_params = best_M_params.iloc[range(0, len(best_M_params), 5)]
        best_M_params.to_csv(f'results/optimal_params_M_gaussian_{target_instances}_{d}.csv')



In [ ]:
# Here we find the optimal settings for slash (and each no. of target instances and d-value)
config_cols = ['v', 'source_tree_size', 'target_tree_size', 'm_0', 'k']
target_instances_list = [100, 300, 500]
d_list = [3,6,9]
for target_instances in target_instances_list:
    for d in d_list:
        LS_data_slash = data_slash[data_slash['method'] == 'LSTransferTreeBoost']
        LS_data_slash = LS_data_slash[(LS_data_slash['d'] == d) & (LS_data_slash['target_instances'] == target_instances)]

        best_LS, best_LS_params = topk_per_d_per_method(
            LS_data_slash,
            config_cols, k=1000
        )
        best_LS_params = best_LS_params.iloc[range(0, len(best_LS_params), 5)]
        best_LS_params.to_csv(f'results/optimal_params_LS_slash_{target_instances}_{d}.csv')

        LAD_data_slash = data_slash[data_slash['method'] == 'LADTransferTreeBoost']
        LAD_data_slash = LAD_data_slash[(LAD_data_slash['d'] == d) & (LAD_data_slash['target_instances'] == target_instances)]

        best_LAD, best_LAD_params = topk_per_d_per_method(
            LAD_data_slash,
            config_cols, k=1000
        )
        best_LAD_params = best_LAD_params.iloc[range(0, len(best_LAD_params), 5)]
        best_LAD_params.to_csv(f'results/optimal_params_LAD_slash_{target_instances}_{d}.csv')

        M_data_slash = data_slash[data_slash['method'] == 'MTransferTreeBoost']
        M_data_slash = M_data_slash[(M_data_slash['d'] == d) & (M_data_slash['target_instances'] == target_instances)]

        best_M, best_M_params = topk_per_d_per_method(
            M_data_slash,
            config_cols, k=1000
        )
        best_M_params = best_M_params.iloc[range(0, len(best_M_params), 5)]
        best_M_params.to_csv(f'results/optimal_params_M_slash_{target_instances}_{d}.csv')



In [ ]:
# Here we find the optimal settings for t (and each no. of target instances and d-value)
config_cols = ['v', 'source_tree_size', 'target_tree_size', 'm_0', 'k']
target_instances_list = [100, 300, 500]
d_list = [3,6,9]
for target_instances in target_instances_list:
    for d in d_list:
        LS_data_t = data_t[data_t['method'] == 'LSTransferTreeBoost']
        LS_data_t = LS_data_t[(LS_data_t['d'] == d) & (LS_data_t['target_instances'] == target_instances)]

        best_LS, best_LS_params = topk_per_d_per_method(
            LS_data_t,
            config_cols, k=1000
        )
        best_LS_params = best_LS_params.iloc[range(0, len(best_LS_params), 5)]
        best_LS_params.to_csv(f'results/optimal_params_LS_t_{target_instances}_{d}.csv')

        LAD_data_t = data_t[data_t['method'] == 'LADTransferTreeBoost']
        LAD_data_t = LAD_data_t[(LAD_data_t['d'] == d) & (LAD_data_t['target_instances'] == target_instances)]

        best_LAD, best_LAD_params = topk_per_d_per_method(
            LAD_data_t,
            config_cols, k=1000
        )
        best_LAD_params = best_LAD_params.iloc[range(0, len(best_LAD_params), 5)]
        best_LAD_params.to_csv(f'results/optimal_params_LAD_t_{target_instances}_{d}.csv')

        M_data_t = data_t[data_t['method'] == 'MTransferTreeBoost']
        M_data_t = M_data_t[(M_data_t['d'] == d) & (M_data_t['target_instances'] == target_instances)]

        best_M, best_M_params = topk_per_d_per_method(
            M_data_t,
            config_cols, k=1000
        )
        best_M_params = best_M_params.iloc[range(0, len(best_M_params), 5)]
        best_M_params.to_csv(f'results/optimal_params_M_t_{target_instances}_{d}.csv')



In [ ]:
import pandas as pd
#Create Latex Table

# Hyperparameters to report
config_cols = ['source_tree_size', 'target_tree_size', 'm_0', 'k']
top_n = 5
target_instances_list = [100, 300, 500]
d_list = [3, 6, 9]

# Load all best_params CSVs into a single dict
dfs = {}  # key = (target_instances, d)
for ti in target_instances_list:
    for d in d_list:
        dfs[(ti,d)] = pd.read_csv(f'results/optimal_params_M_slash_{ti}_{d}.csv') #Select error dist here

# Print LaTeX table
print("\\begin{table}[htbp]")
print("    \\centering")
print("    \\caption{Top-5 hyperparameter configurations for each target instance and Gaussian errors for each~$d$.}")
print("    \\begin{tabular}{|c|c|r|r|r|r|r|}")
print("    \\hline")
print("Target instances & $d$ & Max. source tree depth & Max. target tree depth & $m_0$ & $k$ & avg. RMSE \\\\")
print("    \\hline")

for ti in target_instances_list:
    print(f"    \\multirow{{{top_n*len(d_list)}}}{{*}}{{{ti}}} ", end="")
    first_ti = True
    for d in d_list:
        df = dfs[(ti,d)]
        top5 = df.head(top_n)
        for idx, row in top5.iterrows():
            # Print d only for the first row of this d-group
            d_str = f"{d}" if idx == top5.index[0] else ""
            row_str = f"& {d_str} & {int(row['source_tree_size'])} & {int(row['target_tree_size'])} "
            row_str += f"& {round(row['m_0'],1)} & {round(row['k'],2)} "
            row_str += f"& {round(row['avg_rmse'],3)} \\\\"
            print("    " + row_str)
        print("    \\cline{2-7}")
    print("    \\hline")

print("    \\end{tabular}")
print("    \\label{table:optimal-hyperparams}")
print("\\end{table}")